In [ ]:
GIT_REPOSITORY = "https://github.com/draklowell/learnable-wavelets.git"
GIT_COMMIT = "main"

In [ ]:
# Fix for invalid sympy dependency on Google Colab
%pip uninstall -y sympy
%pip install --no-cache-dir -q sympy==1.13.3

In [ ]:
%pip install --no-cache-dir -q "learnable-wavelets @ git+{GIT_REPOSITORY}@{GIT_COMMIT}"

In [ ]:
import torch
import wandb

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

from learnable_wavelets import (
    WaveletTransformParameters,
    WaveletTransformParameters2D,
    WaveletTransformAnalysis2D,
    WaveletTransformSynthesis2D,
    wandb as lw_wandb,
    mse_loss,
    l1_sparsity_loss,
    psnr_metric,
)

In [ ]:
LEARNING_RATE = 1e-3
BATCH_SIZE = 32000
VALIDATION_SIZE = None
NUM_EPOCHS = 2
TAP_SIZE = 8
PADDING_MODE = "reflect"
L1_SPARSITY_WEIGHT = 0.01

IS_CUDA = torch.cuda.is_available()
DEVICE = "cuda" if IS_CUDA else "cpu"
IMAGE_DTYPE = torch.float32
PARAMS_DTYPE = torch.float64

In [ ]:
wandb.init(
    project="learnable-wavelets",
    notes=f"Drop details to 0",
    config={
        "git-repo": GIT_REPOSITORY,
        "git-commit": GIT_COMMIT,
        "learning-rate": LEARNING_RATE,
        "batch-size": BATCH_SIZE,
        "validation-size": VALIDATION_SIZE,
        "num-epochs": NUM_EPOCHS,
        "tap-size": TAP_SIZE,
        "padding-mode": PADDING_MODE,
        "dataset": "MNIST",
        "l1-sparsity-weight": L1_SPARSITY_WEIGHT,
        "image-dtype": str(IMAGE_DTYPE),
        "params-dtype": str(PARAMS_DTYPE),
    },
)

In [ ]:
class WaveletTransform(torch.nn.Module):
    def __init__(
        self,
        support_size: int,
        padding_mode: str = "reflect",
    ):
        super().__init__()
        self.params = WaveletTransformParameters(support_size=support_size)
        self.params2d = WaveletTransformParameters2D()
        self.analysis = WaveletTransformAnalysis2D(padding_mode=padding_mode)
        self.synthesis = WaveletTransformSynthesis2D()

    def forward(self, x):
        filters = self.params()
        filters2d = self.params2d(filters).to(dtype=x.dtype)

        decomposition = list(self.analysis(x, filters2d))

        x_rec = self.synthesis(*decomposition, filters2d)

        for i, component in enumerate(decomposition):
            decomposition[i] = component.view(x.shape[0], x.shape[1], -1)

        return (
            torch.concat(decomposition, dim=-1),
            x_rec[:, :, : x.shape[-2], : x.shape[-1]],
        )

In [ ]:
transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.ConvertImageDtype(IMAGE_DTYPE),
        transforms.Lambda(lambda x: x * 2 - 1),  # Scale to [-1, 1]
    ]
)

train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root="./data", train=False, download=True, transform=transform)
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=IS_CUDA,
)

validation_size = len(test_ds)
if VALIDATION_SIZE is not None:
    validation_size = min(validation_size, VALIDATION_SIZE)

validation_batch = torch.stack(
    [test_ds[i][0] for i in range(validation_size)],
    dim=0,
)
validation_batch = validation_batch.to(DEVICE)

In [ ]:
model = WaveletTransform(
    support_size=TAP_SIZE,
    padding_mode=PADDING_MODE,
).to(device=DEVICE, dtype=PARAMS_DTYPE)
compiled_model = torch.compile(model, mode="max-autotune")

In [ ]:
wandb.watch(compiled_model, log="all", log_freq=2)

In [ ]:
optimizer = torch.optim.Adam(compiled_model.parameters(), lr=LEARNING_RATE)

In [ ]:
def criterion(x_rec, x, coeffs):
    mse = mse_loss(x_rec, x)
    l1 = l1_sparsity_loss(coeffs)
    total = (1 - L1_SPARSITY_WEIGHT) * mse + L1_SPARSITY_WEIGHT * l1
    return total, {
        "mse": mse.item(),
        "l1": l1.item(),
        "loss": total.item(),
    }

In [ ]:
def train_step(model, batch, optimizer):
    model.train()
    x, _ = batch

    x = x.to(DEVICE)

    optimizer.zero_grad()

    coeffs, x_rec = model(x)

    loss, info = criterion(x_rec, x, coeffs)

    loss.backward()
    optimizer.step()

    return info

In [ ]:
def validate_step(model, batch):
    model.eval()

    x = batch.to(DEVICE)

    with torch.no_grad():
        coeffs, x_rec = model(x)

        _, info = criterion(x_rec, x, coeffs)

        psnr = psnr_metric(x_rec, x)

    return {
        **info,
        "psnr": psnr.item(),
    }, (x[0], x_rec[0])

In [ ]:
for epoch in range(NUM_EPOCHS):
    for batch in train_loader:
        stats = train_step(compiled_model, batch, optimizer)

        wandb.log(
            {
                "epoch": epoch,
                "train": stats,
            }
        )

    stats, (image, reconstructed_image) = validate_step(
        compiled_model, validation_batch
    )

    wavelet = lw_wandb.get_wavelet(compiled_model.params().cpu().detach())
    reconstruction = lw_wandb.get_reconstruction(
        image.cpu().detach(),
        reconstructed_image.cpu().detach(),
    )

    wandb.log(
        {
            "validation": {
                **stats,
                "epoch": epoch,
                "wavelet": wavelet,
                "reconstruction": reconstruction,
            }
        }
    )

In [ ]:
code_artifact = wandb.Artifact("notebook", type="notebook")
lw_wandb.add_code_to_artifact(code_artifact)
wandb.log_artifact(code_artifact)

torch.save(model.state_dict(), "model.pt")
model_artifact = wandb.Artifact("model", type="model")
model_artifact.add_file("model.pt")
wandb.log_artifact(model_artifact)

wandb.finish()